In [0]:
# =============================================================================
# F1-Pulse | Bronze Layer — Safe Ingestion
# Notebook: 01_Ingest_F1_Data
# Author:   Jafar891
# Updated:  2025
#
# Ingests raw F1 data from the OpenF1 REST API into Delta Lake Bronze tables.
# Designed for Databricks (Serverless). Idempotent — safe to re-run.
# =============================================================================

import requests
import pandas as pd
import logging
import time
from typing import Optional
from pyspark.sql import DataFrame
from pyspark.sql.functions import current_timestamp, lit

# ---------------------------------------------------------------------------
# Configuration  — change these, never touch the logic below
# ---------------------------------------------------------------------------
YEAR          = 2025
CATALOG       = "f1_project"
SCHEMA        = "bronze"
MAX_RETRIES   = 3
RETRY_DELAY_S = 5        # seconds between retries
REQUEST_TIMEOUT_S = 30   # per HTTP call

BASE_URL = "https://api.openf1.org/v1"

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("f1_pulse.bronze")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def fetch_with_retry(url: str, max_retries: int = MAX_RETRIES) -> Optional[list]:
    """
    GET a URL with exponential-ish back-off retry.
    Returns parsed JSON list or None on failure.
    """
    for attempt in range(1, max_retries + 1):
        try:
            log.info(f"  GET {url}  (attempt {attempt}/{max_retries})")
            response = requests.get(url, timeout=REQUEST_TIMEOUT_S)
            response.raise_for_status()          # raises on 4xx / 5xx
            data = response.json()
            if not isinstance(data, list):
                log.error(f"  Unexpected response format from {url}")
                return None
            log.info(f"  ✅ {len(data)} records received")
            return data

        except requests.exceptions.Timeout:
            log.warning(f"  ⏱ Timeout on attempt {attempt}")
        except requests.exceptions.HTTPError as e:
            log.error(f"  ❌ HTTP error: {e}")
            if response.status_code in (400, 401, 403, 404):
                break                           # non-retryable
        except requests.exceptions.RequestException as e:
            log.warning(f"  ⚠️  Request error on attempt {attempt}: {e}")

        if attempt < max_retries:
            log.info(f"  Retrying in {RETRY_DELAY_S}s …")
            time.sleep(RETRY_DELAY_S)

    log.error(f"  ❌ All {max_retries} attempts failed for {url}")
    return None


def safe_cast_pdf(pdf: pd.DataFrame) -> pd.DataFrame:
    """
    Stringify only object/mixed-type columns so Spark can infer the schema
    without nuking numeric or boolean types.
    """
    for col in pdf.columns:
        if pdf[col].dtype == object:
            pdf[col] = pdf[col].astype(str).replace("None", None)
    return pdf


def pdf_to_spark(data: list, source_url: str) -> Optional[DataFrame]:
    """
    Convert a list of JSON records → Pandas → Spark DataFrame.
    Adds audit columns: ingested_at, source_url.
    Returns None if data is empty or conversion fails.
    """
    if not data:
        log.warning("  Empty dataset — skipping table write.")
        return None
    try:
        pdf = pd.DataFrame(data)
        pdf = safe_cast_pdf(pdf)
        df = spark.createDataFrame(pdf)  # noqa: F821  (spark injected by Databricks)
        df = (
            df.withColumn("ingested_at", current_timestamp())
              .withColumn("source_url", lit(source_url))
        )
        return df
    except Exception as e:
        log.error(f"  ❌ DataFrame conversion failed: {e}")
        return None


def write_bronze(df: DataFrame, table_name: str) -> None:
    """
    Persist a DataFrame to the Bronze Delta table.
    Uses overwrite + schema evolution to keep reruns idempotent.
    """
    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    try:
        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")   # handles schema drift
              .saveAsTable(full_table)
        )
        log.info(f"  ✅ Written → {full_table}  ({df.count()} rows)")
    except Exception as e:
        log.error(f"  ❌ Failed to write {full_table}: {e}")
        raise


def get_latest_race_session(session_data: list) -> Optional[dict]:
    """
    Safely extract the latest race session from the sessions list.
    Prefers the last session whose session_type == 'Race'.
    Falls back to session_data[-1] if no typed match found.
    """
    race_sessions = [s for s in session_data if s.get("session_type") == "Race"]
    if race_sessions:
        return race_sessions[-1]
    log.warning("  No session_type='Race' found — falling back to last record.")
    return session_data[-1] if session_data else None


# ---------------------------------------------------------------------------
# Main ingestion
# ---------------------------------------------------------------------------

def ingest_bronze() -> None:
    log.info("=" * 60)
    log.info(f"F1-Pulse Bronze Ingestion — {YEAR}")
    log.info("=" * 60)

    # ------------------------------------------------------------------
    # 1. Sessions
    # ------------------------------------------------------------------
    sessions_url = f"{BASE_URL}/sessions?year={YEAR}&session_type=Race"
    log.info(f"\n[1/5] Ingesting sessions ({YEAR}) …")

    session_data = fetch_with_retry(sessions_url)
    if not session_data:
        raise RuntimeError("Cannot proceed — sessions endpoint failed.")

    df_sessions = pdf_to_spark(session_data, sessions_url)
    if df_sessions:
        write_bronze(df_sessions, f"raw_sessions_{YEAR}")

    # ------------------------------------------------------------------
    # 2. Resolve latest race session key
    # ------------------------------------------------------------------
    latest_session = get_latest_race_session(session_data)
    if not latest_session:
        raise RuntimeError("Could not determine latest race session.")

    session_key = latest_session["session_key"]
    session_name = latest_session.get("session_name", "Unknown")
    log.info(f"\n  Latest race session → key={session_key}  name={session_name}")

    # ------------------------------------------------------------------
    # 3. Race endpoints
    # ------------------------------------------------------------------
    race_endpoints = {
        f"raw_drivers_{YEAR}": (
            f"{BASE_URL}/drivers?session_key={session_key}"
        ),
        f"raw_laps_{YEAR}": (
            f"{BASE_URL}/laps?session_key={session_key}"
        ),
        f"raw_telemetry_{YEAR}": (
            f"{BASE_URL}/car_data?session_key={session_key}&speed>250"
        ),
    }

    results = {"success": [], "failed": []}

    for i, (table_name, url) in enumerate(race_endpoints.items(), start=2):
        log.info(f"\n[{i+1}/5] Ingesting {table_name} …")
        data = fetch_with_retry(url)
        if data is None:
            results["failed"].append(table_name)
            continue

        df = pdf_to_spark(data, url)
        if df is None:
            results["failed"].append(table_name)
            continue

        try:
            write_bronze(df, table_name)
            results["success"].append(table_name)
        except Exception:
            results["failed"].append(table_name)

    # ------------------------------------------------------------------
    # 4. Summary
    # ------------------------------------------------------------------
    log.info("\n" + "=" * 60)
    log.info("INGESTION SUMMARY")
    log.info("=" * 60)
    log.info(f"  Session key : {session_key} ({session_name})")
    log.info(f"  ✅ Success  : {len(results['success'])} tables")
    for t in results["success"]:
        log.info(f"      → {CATALOG}.{SCHEMA}.{t}")
    if results["failed"]:
        log.warning(f"  ❌ Failed   : {len(results['failed'])} tables")
        for t in results["failed"]:
            log.warning(f"      → {t}")
    log.info("=" * 60)

    if results["failed"]:
        raise RuntimeError(
            f"Ingestion completed with errors: {results['failed']}"
        )


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
ingest_bronze()

2026-04-08 20:00:19 [INFO] ============================================================
2026-04-08 20:00:19 [INFO] F1-Pulse Bronze Ingestion — 2025
2026-04-08 20:00:19 [INFO] ============================================================
2026-04-08 20:00:19 [INFO] 
[1/5] Ingesting sessions (2025) …
2026-04-08 20:00:19 [INFO]   GET https://api.openf1.org/v1/sessions?year=2025&session_type=Race  (attempt 1/3)
2026-04-08 20:00:19 [INFO]   ✅ 30 records received
2026-04-08 20:01:13 [INFO]   ✅ Written → f1_project.bronze.raw_sessions_2025  (30 rows)
2026-04-08 20:01:13 [INFO] 
  Latest race session → key=9839  name=Race
2026-04-08 20:01:13 [INFO] 
[3/5] Ingesting raw_drivers_2025 …
2026-04-08 20:01:13 [INFO]   GET https://api.openf1.org/v1/drivers?session_key=9839  (attempt 1/3)
2026-04-08 20:01:14 [INFO]   ✅ 20 records received
2026-04-08 20:01:19 [INFO]   ✅ Written → f1_project.bronze.raw_drivers_2025  (20 rows)
2026-04-08 20:01:19 [INFO] 
[4/5] Ingesting raw_laps_2025 …
2026-04-08 20:01:19 